# UAS Data Mining — BMI Analysis & Klasifikasi Obesitas

**Metode:** Random Forest Classifier + KMeans Clustering + SHAP Explainable AI  
**Framework:** CRISP-DM  
**Dataset:** Age Weight Height BMI Analysis (Kaggle) — 741 record, 5 atribut  



# 1. Import Library

Mengimpor semua library yang dibutuhkan untuk analisis data, visualisasi, pemodelan machine learning, dan penyimpanan model.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
import os
import json
import joblib
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.cluster import KMeans
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    silhouette_score
)

plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('husl')
print('✅ Library berhasil diimport!')

In [ ]:
# Buat folder output jika belum ada
os.makedirs('assets', exist_ok=True)
os.makedirs('model', exist_ok=True)
print('✅ Folder assets/ dan model/ siap.')

# 2. Data Understanding

 Dataset bersumber dari Kaggle dengan 741 record dan 5 atribut: Age, Height, Weight, Bmi, dan BmiClass.

In [ ]:
# Baca dataset dari folder dataset/
# Jalankan notebook dari folder notebook/, maka path relatifnya adalah ../dataset/
df = pd.read_csv('../dataset/datasetbmi.csv')

print('Shape dataset:', df.shape)
print('\nTipe data:')
print(df.dtypes)
print('\nPreview data:')
df.head()

In [ ]:
print('=== Statistik Deskriptif ===')
df.describe().round(3)

In [ ]:
print('=== Missing Values ===')
print(df.isnull().sum())
print('\nTotal missing:', df.isnull().sum().sum())
print('\n=== Distribusi BmiClass ===')
print(df['BmiClass'].value_counts())

# 3. Exploratory Data Analysis (EDA)

EDA dilakukan untuk memahami distribusi setiap fitur, hubungan antar variabel, serta mendeteksi keberadaan outlier. Visualisasi meliputi histogram distribusi, bar chart kategori, heatmap korelasi, dan boxplot.

In [ ]:
# Distribusi fitur numerik
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle('Distribusi Fitur Numerik', fontsize=16, fontweight='bold')

features = ['Age', 'Height', 'Weight', 'Bmi']
colors = ['#2196F3', '#4CAF50', '#FF9800', '#E91E63']

for ax, feat, color in zip(axes.flat, features, colors):
    sns.histplot(df[feat], kde=True, ax=ax, color=color, alpha=0.7)
    ax.set_title(f'Distribusi {feat}', fontweight='bold')
    ax.set_xlabel(feat)
    ax.set_ylabel('Frekuensi')

plt.tight_layout()
plt.savefig('assets/distribusi_fitur.png', dpi=100, bbox_inches='tight')
plt.show()
print('✅ Gambar distribusi_fitur.png disimpan!')

In [ ]:
# Bar chart distribusi kategori + scatter plot Weight vs Height
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

bmi_counts = df['BmiClass'].value_counts()
bars = axes[0].bar(bmi_counts.index, bmi_counts.values,
                   color=['#2196F3','#4CAF50','#FF9800','#E91E63','#9C27B0','#00BCD4'])
axes[0].set_title('Distribusi Kategori BmiClass', fontweight='bold')
axes[0].set_xlabel('Kategori')
axes[0].set_ylabel('Jumlah')
axes[0].tick_params(axis='x', rotation=30)
for bar, val in zip(bars, bmi_counts.values):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 2,
                 str(val), ha='center', fontweight='bold')

categories = df['BmiClass'].unique()
palette = sns.color_palette('husl', len(categories))
for cat, color in zip(categories, palette):
    subset = df[df['BmiClass'] == cat]
    axes[1].scatter(subset['Weight'], subset['Height'], label=cat, alpha=0.6, color=color)
axes[1].set_title('Weight vs Height per Kategori', fontweight='bold')
axes[1].set_xlabel('Berat Badan (kg)')
axes[1].set_ylabel('Tinggi Badan (m)')
axes[1].legend(fontsize=8)

plt.tight_layout()
plt.savefig('assets/eda_kategori.png', dpi=100, bbox_inches='tight')
plt.show()

In [ ]:

fig, ax = plt.subplots(figsize=(8, 6))
corr = df[['Age', 'Height', 'Weight', 'Bmi']].corr()
sns.heatmap(corr, annot=True, fmt='.3f', cmap='coolwarm',
            square=True, ax=ax, linewidths=0.5)
ax.set_title('Correlation Matrix', fontweight='bold', fontsize=14)
plt.tight_layout()
plt.savefig('assets/correlation.png', dpi=100, bbox_inches='tight')
plt.show()
print('Korelasi tertinggi dengan BMI:', corr['Bmi'].drop('Bmi').idxmax())

In [ ]:

fig, axes = plt.subplots(1, 4, figsize=(16, 5))
fig.suptitle('Boxplot - Deteksi Outlier', fontsize=14, fontweight='bold')

for ax, feat in zip(axes, ['Age','Height','Weight','Bmi']):
    df.boxplot(column=feat, ax=ax)
    ax.set_title(feat)

plt.tight_layout()
plt.savefig('assets/boxplot_outlier.png', dpi=100, bbox_inches='tight')
plt.show()

print('=== Jumlah Outlier (IQR Method) ===')
for feat in ['Age', 'Height', 'Weight', 'Bmi']:
    Q1 = df[feat].quantile(0.25)
    Q3 = df[feat].quantile(0.75)
    IQR = Q3 - Q1
    outliers = df[(df[feat] < Q1 - 1.5*IQR) | (df[feat] > Q3 + 1.5*IQR)]
    print(f'{feat}: {len(outliers)} outlier')

# 4. Data Preprocessing

Tahap preprocessing meliputi:
1. Feature engineering
2. Standarisasi fitur 
3. Train-test split 

In [ ]:

def bmi_category(bmi):
    if bmi < 18.5:
        return 'Kurus'
    elif bmi < 25:
        return 'Normal'
    elif bmi < 30:
        return 'Overweight'
    else:
        return 'Obesitas'

df['Category'] = df['Bmi'].apply(bmi_category)

print('Distribusi Category:')
print(df['Category'].value_counts())

# Pilih fitur input dan target
X = df[['Age', 'Height', 'Weight', 'Bmi']]
y = df['Category']

print('\nShape X:', X.shape)
print('Jumlah kelas:', y.nunique(), '->', list(y.unique()))

In [ ]:

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

print('Sebelum scaling - Mean:', X.mean().round(2).values)
print('Sesudah scaling - Mean:', X_scaled.mean(axis=0).round(3))
print('Sesudah scaling - Std :', X_scaled.std(axis=0).round(3))

# Train-test split dengan stratify agar proporsi kelas tetap seimbang
X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y, test_size=0.2, random_state=42, stratify=y
)
print(f'\nData train: {X_train.shape[0]} | Data test: {X_test.shape[0]}')

# 5. Modeling — Classification (Random Forest)

Random Forest dipilih karena merupakan algoritma ensemble yang robust terhadap overfitting, mampu menangani data non-linear, dan memberikan feature importance yang dapat diinterpretasi. Model dilatih dengan 100 pohon keputusan (n_estimators=100).

Evaluasi dilakukan menggunakan:
- Accuracy dan Classification Report (precision, recall, F1-score)
- Cross-Validation 5-Fold untuk estimasi performa yang lebih stabil
- Confusion Matrix untuk melihat distribusi prediksi per kelas

In [ ]:

clf = RandomForestClassifier(n_estimators=100, random_state=42)
clf.fit(X_train, y_train)

y_pred = clf.predict(X_test)
acc = accuracy_score(y_test, y_pred)

print(f'Accuracy: {acc:.4f} ({acc*100:.2f}%)')
print('\nClassification Report:')
print(classification_report(y_test, y_pred))

In [ ]:

cv_scores = cross_val_score(clf, X_scaled, y, cv=5, scoring='accuracy')
print('Cross-Validation 5-Fold:')
print(f'  Scores : {cv_scores.round(4)}')
print(f'  Mean   : {cv_scores.mean():.4f}')
print(f'  Std    : {cv_scores.std():.4f}')

In [ ]:

cm = confusion_matrix(y_test, y_pred, labels=clf.classes_)
fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=clf.classes_, yticklabels=clf.classes_, ax=ax)
ax.set_title('Confusion Matrix - Random Forest', fontweight='bold', fontsize=14)
ax.set_xlabel('Predicted Label')
ax.set_ylabel('True Label')
plt.tight_layout()
plt.savefig('assets/confusion_matrix.png', dpi=100, bbox_inches='tight')
plt.show()

In [ ]:

feat_imp = pd.DataFrame({
    'Feature': ['Age', 'Height', 'Weight', 'Bmi'],
    'Importance': clf.feature_importances_
}).sort_values('Importance', ascending=False)

fig, ax = plt.subplots(figsize=(8, 5))
bars = ax.barh(feat_imp['Feature'], feat_imp['Importance'],
               color=['#E91E63','#FF9800','#4CAF50','#2196F3'])
ax.set_title('Feature Importance - Random Forest', fontweight='bold', fontsize=14)
ax.set_xlabel('Importance Score')
for bar, val in zip(bars, feat_imp['Importance']):
    ax.text(bar.get_width() + 0.001, bar.get_y() + bar.get_height()/2,
            f'{val:.4f}', va='center', fontweight='bold')
plt.tight_layout()
plt.savefig('assets/feature_importance.png', dpi=100, bbox_inches='tight')
plt.show()
print(feat_imp.to_string(index=False))

# 6. Modeling Clustering (KMeans)

KMeans digunakan untuk mengelompokkan populasi berdasarkan karakteristik antropometri tanpa menggunakan label kelas. Jumlah cluster optimal (k=3) ditentukan melalui:
- Elbow Method mencari titik di mana penurunan inertia mulai melambat
- Silhouette Score  mengukur seberapa baik setiap data cocok dengan clusternya sendiri

In [ ]:

inertias = []
silhouettes = []
K_range = range(2, 9)

for k in K_range:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    km.fit(X_scaled)
    inertias.append(km.inertia_)
    sil = silhouette_score(X_scaled, km.labels_)
    silhouettes.append(sil)
    print(f'k={k} | Inertia: {km.inertia_:.2f} | Silhouette: {sil:.4f}')

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(K_range, inertias, 'bo-', linewidth=2, markersize=8)
axes[0].axvline(x=3, color='red', linestyle='--', label='k=3 (dipilih)')
axes[0].set_title('Elbow Method', fontweight='bold', fontsize=14)
axes[0].set_xlabel('Jumlah Cluster (k)')
axes[0].set_ylabel('Inertia (WCSS)')
axes[0].legend()

axes[1].plot(K_range, silhouettes, 'gs-', linewidth=2, markersize=8)
axes[1].axvline(x=3, color='red', linestyle='--', label='k=3 (dipilih)')
axes[1].set_title('Silhouette Score', fontweight='bold', fontsize=14)
axes[1].set_xlabel('Jumlah Cluster (k)')
axes[1].set_ylabel('Silhouette Score')
axes[1].legend()

plt.tight_layout()
plt.savefig('assets/elbow_silhouette.png', dpi=100, bbox_inches='tight')
plt.show()

In [ ]:

kmeans = KMeans(n_clusters=3, random_state=42, n_init=10)
df['Cluster'] = kmeans.fit_predict(X_scaled)

cluster_bmi_mean = df.groupby('Cluster')['Bmi'].mean().sort_values()
label_names = ['Berat Rendah', 'Berat Normal', 'Berat Tinggi']
cluster_labels_map = {int(cid): label_names[i]
                      for i, (cid, _) in enumerate(cluster_bmi_mean.items())}

df['ClusterLabel'] = df['Cluster'].map(cluster_labels_map)

print('Distribusi Cluster:')
print(df['ClusterLabel'].value_counts())
print('\nRata-rata per Cluster:')
print(df.groupby('ClusterLabel')[['Age','Height','Weight','Bmi']].mean().round(2))

In [ ]:

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

for label, color in zip(['Berat Rendah','Berat Normal','Berat Tinggi'],
                        ['#2196F3','#4CAF50','#E91E63']):
    subset = df[df['ClusterLabel'] == label]
    axes[0].scatter(subset['Weight'], subset['Bmi'], label=label, alpha=0.7, color=color, s=40)
    axes[1].scatter(subset['Age'], subset['Bmi'], label=label, alpha=0.7, color=color, s=40)

axes[0].set_title('Clustering: Weight vs BMI', fontweight='bold')
axes[0].set_xlabel('Berat Badan (kg)')
axes[0].set_ylabel('BMI')
axes[0].legend()

axes[1].set_title('Clustering: Age vs BMI', fontweight='bold')
axes[1].set_xlabel('Umur')
axes[1].set_ylabel('BMI')
axes[1].legend()

plt.tight_layout()
plt.savefig('assets/cluster_scatter.png', dpi=100, bbox_inches='tight')
plt.show()

# 7. Explainable AI — SHAP

SHAP (SHapley Additive exPlanations) digunakan untuk menjelaskan output model Random Forest secara transparan. SHAP mengukur kontribusi marginal setiap fitur terhadap prediksi berdasarkan teori permainan (game theory), sehingga model *black-box* menjadi lebih mudah diinterpretasi.



In [ ]:
import shap

explainer = shap.TreeExplainer(clf)
shap_values = explainer.shap_values(X_test)

shap.summary_plot(shap_values, X_test,
                  feature_names=['Age','Height','Weight','Bmi'],
                  class_names=clf.classes_,
                  plot_type='bar', show=False)
plt.tight_layout()
plt.savefig('assets/shap_importance.png', dpi=100, bbox_inches='tight')
plt.show()
print(' SHAP importance plot disimpan!')

# 8. Simpan Model

Model yang sudah dilatih disimpan ke folder model/ menggunakan joblib. File yang disimpan:
- classifier.pkl  model Random Forest
- kmeans.pkl  model KMeans
- scaler.pkl  StandardScaler untuk preprocessing input baru
- cluster_labels.json  mapping cluster ID ke nama label
- class_names.json  daftar nama kelas klasifikasi

In [ ]:

os.makedirs('model', exist_ok=True)

joblib.dump(clf,    os.path.join('model', 'classifier.pkl'))
joblib.dump(kmeans, os.path.join('model', 'kmeans.pkl'))
joblib.dump(scaler, os.path.join('model', 'scaler.pkl'))

with open(os.path.join('model', 'cluster_labels.json'), 'w') as f:
    json.dump(cluster_labels_map, f)

with open(os.path.join('model', 'class_names.json'), 'w') as f:
    json.dump(list(clf.classes_), f)

print('✅ Semua model berhasil disimpan ke folder model/')
print('Files:', os.listdir('model'))

# 9. Kesimpulan

Proses evaluasi model diawali dengan Random Forest Classifier, sebuah model klasifikasi yang berhasil memetakan data ke dalam empat kategori berat badan, yaitu Kurus, Normal, Overweight, dan Obesitas. Untuk memastikan performa model ini tetap stabil saat menghadapi data baru di dunia nyata, kami mengujinya menggunakan metode 5-fold cross-validation (pemvalidatan silang). Dari pengujian ini, ditemukan bahwa fitur Bmi (indeks massa tubuh) dan Weight (berat badan) merupakan dua faktor paling dominan yang paling menentukan hasil prediksi model.Di sisi lain, kami juga menerapkan metode pengelompokan non-opsional menggunakan KMeans Clustering. Berdasarkan analisis kurva Elbow dan penilaian Silhouette Score—dua metode standar untuk mencari jumlah kelompok paling ideal—dipilihlah nilai $k=3$. Angka ini berhasil memisahkan populasi data secara alami menjadi tiga kelompok besar yang sangat jelas, yaitu kelompok kelompok Berat Rendah, Berat Normal, dan Berat Tinggi.Guna memberikan transparansi pada cara kerja model Random Forest yang cenderung rumit dan tertutup (black-box), kami mengintegrasikan teknologi SHAP (Explainable AI). Melalui analisis SHAP ini, dikonfirmasi kembali secara visual bahwa Bmi memang merupakan fitur yang memberikan pengaruh paling besar terhadap keputusan model. Akhirnya, model terbaik berupa Random Forest ini kami bungkus dan implementasikan ke dalam aplikasi web berbasis Streamlit, yang seluruh kode dan tampilannya dapat diakses langsung melalui folder app/.